In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp04
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Get the best available compute device: MPS (Mac) > CUDA > CPU."""
    import torch

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp04/ex_3.py ──
"""
Shared infrastructure for MLFP04 Exercise 3 — Dimensionality Reduction.

Contains: data loading, scaling, common output directory, KMeans-based
silhouette evaluation in the embedding space. Technique-specific code
(PCA/KPCA/t-SNE/UMAP algorithms and their plots) lives in the per-
technique files, NOT here.

"""

from pathlib import Path

import numpy as np
import polars as pl
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

from kailash_ml.interop import to_sklearn_input


# ════════════════════════════════════════════════════════════════════════
# OUTPUT + REPRODUCIBILITY
# ════════════════════════════════════════════════════════════════════════

OUTPUT_DIR = Path("outputs") / "ex3_dimreduce"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
DEFAULT_N_CLUSTERS = 4

# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — E-commerce customers (reused from MLFP03)
# ════════════════════════════════════════════════════════════════════════


def load_customer_matrix() -> tuple[np.ndarray, list[str], pl.DataFrame]:
    """Load e-commerce customers, standardise numeric features.

    Returns:
        X          : (n_samples, n_features) standardised float matrix
        feature_cols: list of feature column names in order
        df_raw     : the raw polars DataFrame before scaling
    """
    loader = MLFPDataLoader()
    customers = loader.load("mlfp03", "ecommerce_customers.parquet")

    feature_cols = [
        c
        for c, d in zip(customers.columns, customers.dtypes)
        if d in (pl.Float64, pl.Float32, pl.Int64, pl.Int32)
        and c not in ("customer_id",)
    ]

    df_clean = customers.drop_nulls(subset=feature_cols)
    X_raw, _, _ = to_sklearn_input(df_clean, feature_columns=feature_cols)

    scaler = StandardScaler()
    X = scaler.fit_transform(X_raw)
    return X, feature_cols, df_clean


# ════════════════════════════════════════════════════════════════════════
# EMBEDDING-SPACE CLUSTER QUALITY
# ════════════════════════════════════════════════════════════════════════


def evaluate_embedding_silhouette(
    embedding: np.ndarray,
    n_clusters: int = DEFAULT_N_CLUSTERS,
    random_state: int = RANDOM_STATE,
) -> float:
    """Fit KMeans in the embedding space and return the silhouette score.

    This is the standard "does the reducer preserve structure?" probe used
    across all five technique files. Returns -1.0 when only one cluster is
    found (e.g. collapsed embedding).
    """
    km = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=5)
    labels = km.fit_predict(embedding)
    if len(set(labels)) < 2:
        return -1.0
    return float(silhouette_score(embedding, labels))


# ════════════════════════════════════════════════════════════════════════
# SUBSAMPLING — used by KPCA / t-SNE / UMAP / Isomap for kernel-cost paths
# ════════════════════════════════════════════════════════════════════════


def subsample_indices(
    n_samples: int, n_target: int, random_state: int = RANDOM_STATE
) -> np.ndarray:
    """Deterministic subsample indices for expensive O(n^2) methods."""
    rng = np.random.default_rng(random_state)
    return rng.choice(n_samples, min(n_target, n_samples), replace=False)




# ════════════════════════════════════════════════════════════════════════
# MLFP04 — Exercise 3.4: UMAP for production dim reduction
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Understand UMAP's fuzzy topological formulation vs t-SNE
#   - Tune n_neighbors (local vs global) and min_dist (tight vs spread)
#   - Use .transform() for out-of-sample embedding — the production path
#   - Choose UMAP over t-SNE when feature extraction is the goal
#
# PREREQUISITES: 01_pca.py, 03_tsne.py.
#
# ESTIMATED TIME: ~30 min
#
# TASKS:
#   1. Theory — UMAP as a weighted k-NN graph layout
#   2. Build — fit UMAP with 6 hyperparameter configurations
#   3. Train — fit on subsample, .transform() full dataset (OOS)
#   4. Visualise — silhouette across configurations + 2D scatter
#   5. Apply — MAS (Monetary Authority of Singapore) AML anomaly ranking
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import time

from sklearn.decomposition import PCA

from kailash_ml import ModelVisualizer


# UMAP is an optional extra — fall back gracefully so the exercise stays
# runnable on machines without umap-learn installed (e.g. Colab cold
# start). See rules/dependencies.md "Optional Extras with Loud Failure".
try:
    import umap as umap_lib  # type: ignore

    UMAP_AVAILABLE = True
except ImportError:  # pragma: no cover
    umap_lib = None
    UMAP_AVAILABLE = False
    print("[warn] umap-learn not installed — install with: pip install umap-learn")
    print("       Falling back to PCA 2D for the APPLY phase only.")



## THEORY — UMAP in one paragraph


In [ ]:
# UMAP models the data as a weighted k-NN graph, then optimises a
# low-dimensional layout whose own k-NN graph matches the high-dim one.
# Compared to t-SNE:
#   + preserves BOTH local neighbours AND the global skeleton
#   + supports .transform() for new points (trained embedder becomes
#     a function from R^p to R^2)
#   + faster: ~O(n) amortised, scales to ~1M rows
#   + embeds into arbitrary dimensions, not just 2D
#
# Two key hyperparameters:
#   - n_neighbors: size of the local neighbourhood. Small (5) = local
#     detail, large (50) = global structure.
#   - min_dist: minimum distance between points in the embedding. Small
#     (0.0) = tight clusters, large (1.0) = spread out for visualisation.



## TASK 2 — BUILD: data + PCA pre-reduction


In [ ]:
X, feature_cols, _ = load_customer_matrix()
n_samples, n_features = X.shape

pca_pre = PCA(n_components=min(10, n_features), random_state=42)
X_pca = pca_pre.fit_transform(X)

# Fit subsample; transform the full dataset to showcase OOS.
fit_idx = subsample_indices(n_samples, n_target=3000)
print(f"=== UMAP inputs ===")
print(f"  fit on  : {len(fit_idx):,} rows")
print(f"  transform: {n_samples:,} rows (full dataset, out-of-sample)")



## TASK 3 — TRAIN: sweep n_neighbors x min_dist


In [ ]:
umap_configs = [
    {"n_neighbors": 5, "min_dist": 0.1, "label": "local (n=5, d=0.1)"},
    {"n_neighbors": 15, "min_dist": 0.1, "label": "default (n=15, d=0.1)"},
    {"n_neighbors": 30, "min_dist": 0.1, "label": "broad (n=30, d=0.1)"},
    {"n_neighbors": 15, "min_dist": 0.0, "label": "tight (n=15, d=0.0)"},
    {"n_neighbors": 15, "min_dist": 0.5, "label": "spread (n=15, d=0.5)"},
    {"n_neighbors": 50, "min_dist": 0.5, "label": "global (n=50, d=0.5)"},
]

umap_results: dict[str, dict] = {}

print(f"\n=== UMAP hyperparameter sweep ===")
print(f"{'config':<28}{'silhouette':>14}{'time (s)':>12}")
print("-" * 54)

if UMAP_AVAILABLE:
    for cfg in umap_configs:
        t0 = time.time()
        reducer = umap_lib.UMAP(
            n_components=2,
            n_neighbors=cfg["n_neighbors"],
            min_dist=cfg["min_dist"],
            random_state=42,
            metric="euclidean",
        )
        reducer.fit(X_pca[fit_idx])
        embedding_full = reducer.transform(X_pca)  # out-of-sample for all rows
        elapsed = time.time() - t0

        sil = evaluate_embedding_silhouette(embedding_full)
        umap_results[cfg["label"]] = {
            "embedding": embedding_full,
            "silhouette": sil,
            "time_s": elapsed,
        }
        print(f"{cfg['label']:<28}{sil:>14.4f}{elapsed:>11.1f}")
else:
    # PCA 2D fallback — keeps the exercise runnable in minimal envs.
    pca_2d = PCA(n_components=2, random_state=42)
    embedding_full = pca_2d.fit_transform(X_pca)
    sil = evaluate_embedding_silhouette(embedding_full)
    umap_results["PCA-2D-fallback"] = {
        "embedding": embedding_full,
        "silhouette": sil,
        "time_s": 0.0,
    }
    print(f"{'PCA-2D-fallback':<28}{sil:>14.4f}{0.0:>11.1f}")



### Checkpoint 1


In [ ]:
assert len(umap_results) >= 1, "Must produce at least one UMAP result"
for label, res in umap_results.items():
    assert res["embedding"].shape == (n_samples, 2), (
        f"UMAP {label} must return full-dataset 2D embedding "
        f"(out-of-sample transform)"
    )
print("\n[ok] Checkpoint 1 — out-of-sample transform produced full-dataset 2D")



## TASK 4 — VISUALISE: silhouette across configurations


In [ ]:
viz = ModelVisualizer()
fig = viz.metric_comparison(
    {label: {"Silhouette": r["silhouette"]} for label, r in umap_results.items()}
)
fig.update_layout(title="UMAP: silhouette across hyperparameter configurations")
umap_path = OUTPUT_DIR / "04_umap_sweep.html"
fig.write_html(str(umap_path))
print(f"\nSaved: {umap_path}")

print("\nUMAP hyperparameter guide:")
print("  n_neighbors small -> fine local detail, fractured clusters")
print("  n_neighbors large -> smoother, more global structure")
print("  min_dist   small -> tight clusters (good for downstream KMeans)")
print("  min_dist   large -> spread out (good for visual inspection)")
print("\nOut-of-sample recipe: reducer.fit(train); reducer.transform(new_X)")



## TASK 5 — APPLY: MAS (Monetary Authority of Singapore) AML anomaly ranking


In [ ]:
# SCENARIO: Singapore banks submit suspicious transaction reports (STRs)
# to the MAS COSMIC platform. An AML analytics team embeds every reporting
# entity into a 2D UMAP space built from ~60 features (transaction
# velocity, counterparty diversity, cross-border ratio, cash intensity,
# sector code, anomaly z-scores, device-fingerprint counts). The space is
# refreshed WEEKLY from a stable training slice, and the embedder is then
# applied every NIGHT to the latest transaction rollup for incremental
# screening.
#
# WHY UMAP IS THE RIGHT TOOL:
#   - Out-of-sample .transform() — this is the decisive property here.
#     t-SNE would require a full refit every night, which both changes
#     the axes (breaking the analyst's mental map) and blows the nightly
#     SLA. UMAP fits once per week, then .transform() is ~O(log n) per
#     new point.
#   - Preserves both LOCAL (two entities that behave similarly end up
#     near each other) and GLOBAL (the "high cash velocity" region
#     stays in the same corner week after week) structure.
#   - Scales to ~500K reporting entities without subsampling.
#
# BUSINESS IMPACT: An MAS 2024 financial stability review noted that
# pattern-based screening over entity embeddings raised the positive
# predictive value of STR review from ~11% to ~23% — roughly halving
# false positives. Each false positive costs ~S$1,200 in analyst time
# at MAS + bank compliance. On ~18,000 STRs/yr that's ~S$21.6M/yr in
# avoided triage cost. UMAP refit runs in under an hour on commodity
# hardware; nightly transform is ~3 minutes per 20K new entities.
#
# WHY NOT t-SNE HERE: t-SNE forbids new points without a refit, which
# would reshuffle the axes every night. The analyst's mental map ("fraud
# rings live in the upper-left quadrant") would reset weekly, destroying
# institutional knowledge. UMAP's frozen-embedder workflow preserves that
# map for the life of the weekly fit.

if UMAP_AVAILABLE and umap_results:
    best_label, best = max(umap_results.items(), key=lambda kv: kv[1]["silhouette"])
    print(f"\n=== MAS-style AML projection (UMAP) ===")
    print(f"  Best config     : {best_label}")
    print(f"  Silhouette      : {best['silhouette']:.4f}")
    print(f"  Fit wall time   : {best['time_s']:.1f}s")
    print(f"  Output shape    : {best['embedding'].shape}")
else:
    print("\n[note] Install umap-learn to run the full MAS scenario.")



## REFLECTION


[x] Fit UMAP on a training subsample and transformed the full dataset
      out-of-sample — the production workflow
  [x] Swept n_neighbors and min_dist across 6 configurations
  [x] Measured silhouette in UMAP space vs t-SNE and Kernel PCA
  [x] Sized UMAP for a weekly MAS AML entity-embedding pipeline

  KEY INSIGHT: The out-of-sample transform is what makes UMAP a real
  dimensionality reducer vs t-SNE's picture generator. If you need to
  embed NEW data without refitting, UMAP is the only method in this
  exercise that can do it without compromise.

  Next: 05_comparison.py pits all five techniques against each other on
  the same silhouette ladder and estimates the intrinsic dimensionality
  of the customer feature space.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

